In [31]:
# ============================================================
# COMPLETE FIX: Use All 2000 Images + Continue from best.pt
# ============================================================
import os
import shutil
import yaml
import cv2
import albumentations as A
from glob import glob
from ultralytics import YOLO
import random


In [32]:
# ============================================================
# STEP 1: Fixed Augmentation (copies all files + augments minority)
# ============================================================
print("=" * 60)
print("STEP 1: Augmentation Setup")
print("=" * 60)

aug_pipeline = A.Compose([
    A.RandomBrightnessContrast(p=0.4),
    A.HueSaturationValue(p=0.3),
    A.RGBShift(p=0.2),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=20, p=0.3),
    A.MotionBlur(p=0.2),
])

# Input directories (read-only)
input_labels_dir = "/kaggle/input/jplaurel/labels/Train"
input_images_dir = "/kaggle/input/jplaurel/images/Train"

# Output directories (writable) - use new folder to avoid conflicts
output_labels_dir = "/kaggle/working/dataset_full/labels/Train"
output_images_dir = "/kaggle/working/dataset_full/images/Train"

# Create output directories
os.makedirs(output_labels_dir, exist_ok=True)
os.makedirs(output_images_dir, exist_ok=True)

# Copy ALL original files first
print("\n📋 Copying ALL original files...")
original_labels = glob(os.path.join(input_labels_dir, "*.txt"))
original_images = glob(os.path.join(input_images_dir, "*.jpg")) + glob(os.path.join(input_images_dir, "*.png"))

for txt_file in original_labels:
    shutil.copy(txt_file, output_labels_dir)

for img_file in original_images:
    shutil.copy(img_file, output_images_dir)

print(f"✅ Copied {len(original_labels)} labels and {len(original_images)} images")

# Find minority class samples
minority_ids = [1, 3, 4, 9, 11]
minority_files = []

for txt_file in original_labels:
    with open(txt_file, "r") as f:
        lines = f.readlines()
    if any(int(line.split()[0]) in minority_ids for line in lines if line.strip()):
        minority_files.append(txt_file)

print(f"🎯 Found {len(minority_files)} samples with minority classes")

# Augment minority samples
augmented_count = 0
for txt_file in minority_files:
    base_name = os.path.basename(txt_file).replace(".txt", "")
    img_file = os.path.join(input_images_dir, base_name + ".jpg")
    
    if not os.path.exists(img_file):
        img_file = os.path.join(input_images_dir, base_name + ".png")
    
    if not os.path.exists(img_file):
        continue
    
    image = cv2.imread(img_file)
    if image is None:
        continue
    
    ext = ".jpg" if img_file.endswith(".jpg") else ".png"
    
    for i in range(2):  # 2 augmentations per minority sample
        aug = aug_pipeline(image=image)
        aug_img = aug["image"]
        
        aug_img_name = f"{base_name}_aug{i}{ext}"
        aug_txt_name = f"{base_name}_aug{i}.txt"
        
        cv2.imwrite(os.path.join(output_images_dir, aug_img_name), aug_img)
        shutil.copy(txt_file, os.path.join(output_labels_dir, aug_txt_name))
        augmented_count += 1

print(f"✅ Created {augmented_count} augmented samples")

final_images = len(glob(os.path.join(output_images_dir, "*")))
final_labels = len(glob(os.path.join(output_labels_dir, "*")))
print(f"\n📊 Total before split:")
print(f"   Images: {final_images}")
print(f"   Labels: {final_labels}")

STEP 1: Augmentation Setup

📋 Copying ALL original files...
✅ Copied 2004 labels and 2004 images
🎯 Found 256 samples with minority classes
✅ Created 512 augmented samples

📊 Total before split:
   Images: 2516
   Labels: 2516


In [52]:
# ============================================================
# STEP 3: Create YAML Config
# ============================================================
print("\n" + "=" * 60)
print("STEP 3: Creating YAML Config")
print("=" * 60)

yaml_content = {
    'path': '/kaggle/working/dataset_full',
    'train': '/kaggle/working/dataset_full/images/Train',
    'val': '/kaggle/working/dataset_full/images/Val',
    'names': {
        0: 'sedan',
        1: 'bus',
        2: 'motorcycle',
        3: 'jeepney(sarao)',
        4: 'truck',
        5: 'medium_sized',
        6: 'jeepney(uso-uso)',
        7: 'jeepney(multicab)',
        8: 'van',
        9: 'autorickshaw',
        10: 'tricycle',
        11: 'compact vehicle'
    },
    'nc': 12
}

yaml_path = "/kaggle/working/data_full.yaml"
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print(f"✅ YAML created at: {yaml_path}")


STEP 3: Creating YAML Config
✅ YAML created at: /kaggle/working/data_full.yaml


In [23]:
# ============================================================
# 4. Augmentation setup (only for minority classes by ID)
# ============================================================
import albumentations as A
import cv2
from glob import glob
import shutil
import os

aug_pipeline = A.Compose([
    A.RandomBrightnessContrast(p=0.4),
    A.HueSaturationValue(p=0.3),
    A.RGBShift(p=0.2),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=20, p=0.3),
    A.MotionBlur(p=0.2),
])

# Input directories (read-only)
input_labels_dir = "/kaggle/input/jplaurel/labels/Train"
input_images_dir = "/kaggle/input/jplaurel/images/Train"

# Output directories (writable)
output_labels_dir = "/kaggle/working/labels/Train"
output_images_dir = "/kaggle/working/images/Train"

# Create output directories
os.makedirs(output_labels_dir, exist_ok=True)
os.makedirs(output_images_dir, exist_ok=True)

# First, copy all original files to working directory
print("Copying original files to working directory...")
for txt_file in glob(os.path.join(input_labels_dir, "*.txt")):
    shutil.copy(txt_file, output_labels_dir)

for img_file in glob(os.path.join(input_images_dir, "*")):
    shutil.copy(img_file, output_images_dir)

print("✅ Original files copied")

# minority class IDs based on your mapping
minority_ids = [1, 3, 4, 9, 11]

txt_files = glob(os.path.join(input_labels_dir, "*.txt"))
minority_files = []

for txt_file in txt_files:
    with open(txt_file, "r") as f:
        lines = f.readlines()
    # Check if this label file has at least one minority class
    if any(int(line.split()[0]) in minority_ids for line in lines):
        minority_files.append(txt_file)

print(f"Found {len(minority_files)} training samples containing minority classes. Applying augmentation...")

for txt_file in minority_files:
    img_file = os.path.join(input_images_dir, os.path.basename(txt_file).replace(".txt", ".jpg"))
    if not os.path.exists(img_file):
        img_file = img_file.replace(".jpg", ".png")  # fallback
    
    image = cv2.imread(img_file)
    if image is None:
        continue
    
    for i in range(2):  # 2 augmentations per minority sample
        aug = aug_pipeline(image=image)
        aug_img = aug["image"]
        
        # new filenames
        base_name = os.path.basename(img_file)
        if base_name.endswith(".jpg"):
            aug_name = base_name.replace(".jpg", f"_aug{i}.jpg")
        else:
            aug_name = base_name.replace(".png", f"_aug{i}.png")
        
        # save augmented image to working directory
        cv2.imwrite(os.path.join(output_images_dir, aug_name), aug_img)
        
        # copy label to working directory with new name
        aug_txt_name = aug_name.replace(".jpg", ".txt").replace(".png", ".txt")
        shutil.copy(txt_file, os.path.join(output_labels_dir, aug_txt_name))

print("✅ Augmentation complete for minority classes only")
print(f"📁 Augmented dataset location: /kaggle/working/")
print(f"   - Images: {output_images_dir}")
print(f"   - Labels: {output_labels_dir}")

Copying original files to working directory...
✅ Original files copied
Found 256 training samples containing minority classes. Applying augmentation...
✅ Augmentation complete for minority classes only
📁 Augmented dataset location: /kaggle/working/
   - Images: /kaggle/working/images/Train
   - Labels: /kaggle/working/labels/Train


In [29]:
# ============================================================
# 5. Train YOLOv8
# ============================================================
model = YOLO("yolov8m.pt")  # medium model for balance between speed & accuracy

# Training settings optimized for 16GB VRAM
model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,           # Changed from auto - 16GB can handle this
    workers=4,          # Increased from 2 for faster data loading
    patience=15,
    project="/kaggle/working",
    name="vehicle-yolov8m",
    cache=True,
    
    # Additional optimizations
    device=0,           # Use GPU explicitly
    amp=True,           # Automatic Mixed Precision for faster training
    
    # Fine-tuning hyperparameters for small dataset
    lr0=0.001,          # Initial learning rate (lower for fine-tuning)
    lrf=0.01,           # Final learning rate factor
    weight_decay=0.0005,
    
    # Augmentation settings (since you already did manual augmentation)
    mosaic=0.5,         # Reduced mosaic (you have manual augs)
    mixup=0.0,          # Disable mixup to avoid over-augmentation
    copy_paste=0.0,     # Disable copy-paste
    
    # Regularization for small dataset
    dropout=0.1,        # Add dropout to prevent overfitting
    
    # Evaluation settings
    val=True,
    plots=True,
    save=True,
    save_period=10,     # Save checkpoint every 10 epochs
    
    # Resume capability
    resume=False,
    exist_ok=False,
)

print("✅ Training complete!")
print(f"📊 Results saved to: /kaggle/working/vehicle-yolov8m/")

Ultralytics 8.3.203 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=0.5, multi_scale=False, name=vehicle-yolov8m3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=15, perspective=0.0, pl

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        969       5751       0.83      0.783      0.833      0.759
                 sedan        244        349      0.924      0.871      0.959      0.917
                   bus         23         25      0.622        0.4      0.398      0.319
            motorcycle        682       1857      0.905      0.857      0.936      0.866
        jeepney(sarao)         39         39      0.798      0.718      0.785      0.686
                 truck        148        149      0.689      0.506      0.589      0.478
          medium_sized        772       1477      0.907      0.876       0.95      0.896
      jeepney(uso-uso)        844        892      0.934      0.903      0.966      0.946
     jeepney(multicab)        458        796      0.946      0.898      0.962      0.932
                   van         14         14      0.771          1      0.926      0.871
              tricycle         16         16      0.927      0.938      0.946      0.767
             hatchbac

In [50]:
# ============================================================
# DIAGNOSIS & FIX: Check and recreate the Val split
# ============================================================
from glob import glob
import shutil
import os
import random

# Check current state
train_dir = "/kaggle/working/dataset_full/images/Train"
val_dir = "/kaggle/working/dataset_full/images/Val"
train_labels_dir = "/kaggle/working/dataset_full/labels/Train"
val_labels_dir = "/kaggle/working/dataset_full/labels/Val"

print("📊 Current state:")
print(f"Train images: {len(glob(f'{train_dir}/*'))}")
print(f"Val images: {len(glob(f'{val_dir}/*'))}")

# Create Val directory if missing
os.makedirs(val_dir, exist_ok=True)
os.makedirs(val_labels_dir, exist_ok=True)

# Get all training images and create validation split
all_train_images = glob(f"{train_dir}/*")
random.seed(42)
random.shuffle(all_train_images)

# Split 15% for validation
val_split_count = int(len(all_train_images) * 0.15)
images_to_move = all_train_images[:val_split_count]

print(f"\n🔄 Moving {val_split_count} images from Train to Val...")

moved_count = 0
for img_path in images_to_move:
    img_name = os.path.basename(img_path)
    
    # Determine label name
    if img_name.endswith('.jpg'):
        label_name = img_name.replace('.jpg', '.txt')
    else:
        label_name = img_name.replace('.png', '.txt')
    
    label_path = os.path.join(train_labels_dir, label_name)
    
    # Move image
    try:
        shutil.move(img_path, os.path.join(val_dir, img_name))
        # Move corresponding label if exists
        if os.path.exists(label_path):
            shutil.move(label_path, os.path.join(val_labels_dir, label_name))
        moved_count += 1
    except Exception as e:
        print(f"Error moving {img_name}: {e}")

print(f"✅ Moved {moved_count} image-label pairs")

# Verify final counts
final_train = len(glob(f"{train_dir}/*"))
final_val = len(glob(f"{val_dir}/*"))
final_train_labels = len(glob(f"{train_labels_dir}/*"))
final_val_labels = len(glob(f"{val_labels_dir}/*"))

print(f"\n📊 Final dataset:")
print(f"  Train: {final_train} images, {final_train_labels} labels")
print(f"  Val: {final_val} images, {final_val_labels} labels")
print(f"  Total: {final_train + final_val} images")

if final_val > 0:
    print("\n✅ Dataset is ready! You can now start training.")
else:
    print("\n❌ Val folder is still empty. Check for errors above.")

📊 Current state:
Train images: 2139
Val images: 377

🔄 Moving 320 images from Train to Val...
✅ Moved 320 image-label pairs

📊 Final dataset:
  Train: 1819 images, 1819 labels
  Val: 697 images, 697 labels
  Total: 2516 images

✅ Dataset is ready! You can now start training.


In [3]:
import os
os.chdir("/Users/jfrtenebroso/Developer/LaravelDevelopment/Thesis")
print(f"Changed to: {os.getcwd()}")

# Then use relative paths
model = YOLO("model/best.pt")
model.train(data="model/data.yaml")

Changed to: /Users/jfrtenebroso/Developer/LaravelDevelopment/Thesis
New https://pypi.org/project/ultralytics/8.3.204 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.198 🚀 Python-3.11.13 torch-2.8.0 CPU (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=model/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=model/best.pt, momentum=0.937, mosaic=1.0, multi_scale=False, na

RuntimeError: Dataset 'model/data.yaml' error ❌ Dataset 'model/data.yaml' images not found, missing path '/Users/jfrtenebroso/LaravelDevelopment/Thesis/annotated_videos/images/Val'
Note dataset download directory is '/opt/homebrew/datasets'. You can update this in '/Users/jfrtenebroso/Library/Application Support/Ultralytics/settings.json'

In [5]:
# ============================================================
# NOW START TRAINING (after Val folder is created)
# ============================================================
from ultralytics import YOLO

model = YOLO("/Users/jfrtenebroso/LaravelDevelopment/Thesis/model/best.pt")

model.train(
    data="/Users/jfrtenebroso/LaravelDevelopment/Thesis/model/data.yaml",
    epochs=150,
    imgsz=640,
    batch=8,
    workers=4,
    patience=15,
    project="/kaggle/working",
    name="vehicle-yolov8m-final1",
    cache=True,
    device="mps",
    amp=True,
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    mosaic=0.5,
    mixup=0.0,
    copy_paste=0.0,
    dropout=0.1,
    val=True,
    plots=True,
    save=True,
    save_period=10,
    resume=False,
    exist_ok=False,
)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/jfrtenebroso/LaravelDevelopment/Thesis/model/best.pt'

In [6]:
# ============================================================ 
# DEBUG AND FIX PATHS
# ============================================================
import os
from pathlib import Path
from ultralytics import YOLO

print("🔍 DEBUGGING PATHS:")
print(f"Current working directory: {os.getcwd()}")

# Define all possible paths
model_paths = [
    "/Users/jfrtenebroso/Developer/LaravelDevelopment/Thesis/model/best.pt",
    "./model/best.pt", 
    "../model/best.pt",
    "model/best.pt",
    "best.pt"
]

data_paths = [
    "/Users/jfrtenebroso/Developer/LaravelDevelopment/Thesis/model/data.yaml",
    "./model/data.yaml",
    "../model/data.yaml", 
    "model/data.yaml",
    "data.yaml"
]

# Find the correct model path
model_path = None
for path in model_paths:
    if os.path.exists(path):
        model_path = path
        print(f"✅ Found model at: {path}")
        print(f"   File size: {os.path.getsize(path) / (1024*1024):.1f} MB")
        break
        
if not model_path:
    print("❌ Model not found in any of these locations:")
    for path in model_paths:
        print(f"   - {path} (exists: {os.path.exists(path)})")
    raise FileNotFoundError("Model file not found!")

# Find the correct data path  
data_path = None
for path in data_paths:
    if os.path.exists(path):
        data_path = path
        print(f"✅ Found data.yaml at: {path}")
        break

if not data_path:
    print("❌ data.yaml not found in any location")
    raise FileNotFoundError("data.yaml not found!")

print(f"\n📦 Loading model from: {model_path}")
print(f"📄 Using dataset config: {data_path}")

# Load model
model = YOLO(model_path)
print("✅ Model loaded successfully!")
print(f"Model classes: {list(model.names.values())}")

# ============================================================
# NOW START TRAINING  
# ============================================================
print("\n🚀 Starting training...")

model.train(
    data=data_path,
    epochs=150,
    imgsz=640,
    batch=8,
    workers=4,
    patience=15,
    project="./runs",  # Use relative path for output
    name="vehicle-yolov8m-final1",
    cache=True,
    device="mps", 
    amp=True,
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    mosaic=0.5,
    mixup=0.0,
    copy_paste=0.0,
    dropout=0.1,
    val=True,
    plots=True,
    save=True,
    save_period=10,
    resume=False,
    exist_ok=True,  # Changed to True to avoid conflicts
)

🔍 DEBUGGING PATHS:
Current working directory: /Users/jfrtenebroso/Developer/LaravelDevelopment/Thesis
✅ Found model at: /Users/jfrtenebroso/Developer/LaravelDevelopment/Thesis/model/best.pt
   File size: 148.5 MB
✅ Found data.yaml at: /Users/jfrtenebroso/Developer/LaravelDevelopment/Thesis/model/data.yaml

📦 Loading model from: /Users/jfrtenebroso/Developer/LaravelDevelopment/Thesis/model/best.pt
📄 Using dataset config: /Users/jfrtenebroso/Developer/LaravelDevelopment/Thesis/model/data.yaml
✅ Model loaded successfully!
Model classes: ['sedan', 'bus', 'motorcycle', 'jeepney(sarao)', 'truck', 'medium_sized', 'jeepney(uso-uso)', 'jeepney(multicab)', 'van', 'autorickshaw', 'tricycle', 'compact vehicle']

🚀 Starting training...
New https://pypi.org/project/ultralytics/8.3.204 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.198 🚀 Python-3.11.13 torch-2.8.0 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0

KeyboardInterrupt: 